# Arabic Dialect Toolkit — Colab

تجهيز EGY→MSA، قياس Identity baseline، تدريب mT5، وتقييمه على test.
ارفع نسخة المشروع إلى Google Drive أولًا، ولا ترفع الداتا إلى GitHub.

In [ ]:
%pip install -q "transformers>=4.46" sacrebleu sentencepiece accelerate
import os, sys, subprocess
import torch
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
assert torch.cuda.is_available(), 'فعّل GPU من Runtime > Change runtime type > T4 GPU'

def run(*args):
    """شغّل python -m وعرض الخرج في الـ notebook (subprocess.run العادي ممكن ما يعرضه)."""
    proc = subprocess.Popen([sys.executable, '-m', *args], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')
    assert proc.wait() == 0, f'failed: {args}'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, zipfile

ROOT_NAME = 'Arabic Dialect Identifier + Egyptian-to-MSA Normalizer'
PROJECT_ROOT = f'/content/drive/MyDrive/{ROOT_NAME}'
ZIP_PATH = None  # لو اسم الـ zip مختلف أو في فولدر تاني، حط مساره الكامل هنا

# لو المشروع مش متفك على Drive (أو نسخته قديمة) فك الـ zip هنا. بيتفك مرة واحدة بس.
if not os.path.exists(f'{PROJECT_ROOT}/src/normalizer/evaluate.py'):
    if ZIP_PATH:
        candidates = [ZIP_PATH]
    else:
        found = sorted(glob.glob('/content/drive/MyDrive/*.zip'), key=os.path.getmtime, reverse=True)
        candidates = [p for p in found if any(k in os.path.basename(p) for k in ('Dialect', 'Egyptian', 'Toolkit'))]
    assert candidates, 'ما لقيتش الـ zip في MyDrive. ارفع Arabic_Dialect_Toolkit_full.zip أو حط مساره في ZIP_PATH'
    print('Extracting:', candidates[0])
    with zipfile.ZipFile(candidates[0]) as z:
        z.extractall('/content/drive/MyDrive')

assert os.path.exists(f'{PROJECT_ROOT}/src/normalizer/evaluate.py'), (
    'الـ zip ده نسخة قديمة. امسح الفولدر القديم من Drive وحط مسار Arabic_Dialect_Toolkit_full.zip الجديد في ZIP_PATH')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print('Project:', os.getcwd())

In [ ]:
from pathlib import Path
raw = Path('data/raw/egy_msa_translation')
required = [
    'EGY-parallel_train_60955.txt', 'msa_parallel_train_52659.txt',
    'EGY-parallel_valid_13358.txt', 'msa_parallel_valid_45049.txt',
    'EGY-parallel_test_36402.txt', 'msa_parallel_test_44839.txt'
]
missing = [name for name in required if not (raw / name).exists()]
assert not missing, f'Missing raw files: {missing}'
print('All six parallel files found.')

In [ ]:
# تجهيز splits مع إزالة التكرار وتسريب المصري والفصحى (+ الـ gold set لو موجود)
run('src.normalizer.prepare')
# Identity baseline على test (نفس sacrebleu اللي هيُستخدم للنموذج)
run('src.eval.translation')

In [ ]:
# تدريب mT5 — الحفظ على Drive. fp16 مقفول عمدًا: mT5 ممكن يطلع NaN معاه.
run('src.normalizer.train_seq2seq',
    '--model-name', 'google/mt5-small',
    '--train', 'data/processed/egy_msa/train.jsonl',
    '--valid', 'data/processed/egy_msa/valid.jsonl',
    '--output-dir', 'models/egy_msa_mt5',
    '--epochs', '3', '--batch-size', '8', '--lr', '1e-4')

In [ ]:
# تقييم النموذج على test جنب Identity (نفس المقاييس) — ده اللي بيقول هل النموذج أحسن فعلًا
run('src.normalizer.evaluate', '--model-dir', 'models/egy_msa_mt5')

In [ ]:
run('src.normalizer.predict', '--model-dir', 'models/egy_msa_mt5', '--text', 'انا مش عارف اروح فين')

## Checklist
- تأكد أن Runtime مضبوط على GPU.
- لا تستخدم test أثناء التدريب أو اختيار hyperparameters (اضبط على valid بس).
- قارن النموذج بـ Identity من `reports/mt5_test.json` (نفس الـ backend). لا تقارن بملف `identity_baseline.json` القديم لو `backend` فيه `fallback`.
- بص على `reports/mt5_test_predictions.jsonl` وعلى `copy_rate`: لو عالي، النموذج بينسخ المدخل.
- لا ترفع `data/` أو `models/` إلى GitHub.